# Permian flare sites — folium map

Plots the geolocated VIIRS flare sites with three toggleable views:

- **points by `state` × `in_delaware`** (default) — the NM/TX DiD split
- **footprint polygons** — the real shapes, from `upstream.shp`
- **points by last year detected** — a temporal view tied to the decoupling question

**Data sources (they are not interchangeable):**

| need | file | why |
|---|---|---|
| geometry | `upstream.shp` | clean polygons; the CSV's `geometry` column was truncated to 254 chars by a DBF round-trip |
| rich attributes | `permian_sites_full.csv` | already the Permian subset, with `state`, `in_delaware`, `ID2015…ID2021` |

Join key is the **string** `Catalog ID` (Permian) ↔ `id` (upstream), *not* the integer `ID`
(which is just `upstream`'s positional `index`). Matches 2250 / 2254; the 4 strays are absent from
`upstream` in every format.


## Setup

In [ ]:
# !pip install pyshp folium pandas numpy
import shapefile          # pyshp
import pandas as pd
import numpy as np
import folium

PERMIAN  = "permian_sites_full.csv"
SHP      = "./VNF_multiyear_by_type_2012-2021_v20220822/upstream.shp"          # .shx/.dbf/.prj must sit beside it
ENCODING = "ISO-8859-1"            # from upstream.cpg


## Geometry from the shapefile
Build an `id -> rings` lookup. Coordinates are stored GeoJSON-style `[lon, lat]`.

In [ ]:
sf = shapefile.Reader(SHP, encoding=ENCODING)
fields = [f[0] for f in sf.fields[1:]]
id_pos = fields.index("id")

geom = {}
for shp, rec in zip(sf.shapes(), sf.records()):
    pts = shp.points
    parts = list(shp.parts) + [len(pts)]
    rings = [[[lon, lat] for lon, lat in pts[parts[i]:parts[i+1]]]
             for i in range(len(parts) - 1)]
    geom[rec[id_pos]] = rings

print(f"{len(geom)} polygons read from {SHP}")


## Attributes from the Permian CSV, joined to geometry

In [ ]:
p = pd.read_csv(PERMIAN)
p["rings"] = p["Catalog ID"].map(geom)

matched = p["rings"].notna().sum()
print(f"{matched} / {len(p)} Permian sites matched to a polygon")
print("unmatched (absent from upstream):")
print(p.loc[p["rings"].isna(), ["Catalog ID", "ID", "state"]].to_string(index=False))

p["in_delaware"] = p["in_delaware"].astype(str).str.lower().isin(["true", "1", "yes"])


## Temporal feature: last year detected
From the annual catalog columns `ID2015…ID2021` (non-null = present that year).

In [ ]:
ycols = [c for c in p.columns if c.startswith("ID20")]
yrs   = [int(c[2:]) for c in ycols]

def last_year(row):
    present = [y for y, c in zip(yrs, ycols) if pd.notna(row[c])]
    return max(present) if present else np.nan

p["last_year"] = p.apply(last_year, axis=1)
print(p["last_year"].value_counts(dropna=False).sort_index())
# note: a handful of sites have no annual ID populated (NaN) -> drawn in grey below


## Build the map

Three `FeatureGroup`s under one `LayerControl`. Only the `state × in_delaware` points are on by default;
toggle the polygons (zoom in — each footprint is ~280 m) and the temporal view from the control box.

In [ ]:
pts = p.dropna(subset=["Latitude", "Longitude"]).copy()

fmap = folium.Map(location=[pts["Latitude"].mean(), pts["Longitude"].mean()],
                  zoom_start=7, tiles="CartoDB positron")

# ---- Layer 1: points by state x in_delaware (default) ----
palette = {("NM", True): "#d62728", ("NM", False): "#ff9896",
           ("TX", True): "#1f77b4", ("TX", False): "#aec7e8"}
g1 = folium.FeatureGroup(name="points: state x in_delaware", show=True)
rh = pts["RH_mean"].clip(lower=0).fillna(0)
pts["radius"] = 2 + 6 * np.sqrt(rh / (rh.max() or 1))
for _, r in pts.iterrows():
    col = palette.get((r["state"], bool(r["in_delaware"])), "#777777")
    folium.CircleMarker(
        [r["Latitude"], r["Longitude"]], radius=float(r["radius"]),
        color=col, weight=0.5, fill=True, fill_color=col, fill_opacity=0.6,
        popup=folium.Popup(f"ID {r['ID']}<br>{r['state']} | "
                           f"{'Delaware' if r['in_delaware'] else 'non-Delaware'}<br>"
                           f"RH_mean {r['RH_mean']:.3f}<br>last seen {r['last_year']}",
                           max_width=200),
    ).add_to(g1)
g1.add_to(fmap)

# ---- Layer 2: footprint polygons (real geometry), off by default ----
feats = []
for _, r in pts.iterrows():
    if isinstance(r["rings"], list):
        feats.append({"type": "Feature",
                      "properties": {"state": r["state"], "id": int(r["ID"])},
                      "geometry": {"type": "Polygon", "coordinates": r["rings"]}})
fc = {"type": "FeatureCollection", "features": feats}
state_col = {"NM": "#d62728", "TX": "#1f77b4"}
folium.GeoJson(
    fc, name="footprints (real geometry)", show=False,
    style_function=lambda x: {"color": state_col.get(x["properties"]["state"], "#333"),
                              "weight": 1, "fillOpacity": 0.4},
    tooltip=folium.GeoJsonTooltip(fields=["id", "state"]),
).add_to(fmap)

# ---- Layer 3: points by last year detected, off by default ----
year_col = {2015: "#440154", 2016: "#46327e", 2017: "#365c8d", 2018: "#277f8e",
            2019: "#1fa187", 2020: "#4ac16d", 2021: "#fde725"}
g3 = folium.FeatureGroup(name="points: last year detected", show=False)
for _, r in pts.iterrows():
    col = year_col.get(r["last_year"], "#999999")
    folium.CircleMarker(
        [r["Latitude"], r["Longitude"]], radius=3,
        color=col, weight=0.5, fill=True, fill_color=col, fill_opacity=0.7,
        popup=f"ID {r['ID']} | last seen {r['last_year']}",
    ).add_to(g3)
g3.add_to(fmap)

folium.LayerControl(collapsed=False).add_to(fmap)

legend = '''
<div style="position: fixed; bottom: 24px; left: 24px; z-index: 9999; background: white;
            padding: 10px 12px; border: 1px solid #999; border-radius: 4px;
            font: 12px sans-serif; line-height: 1.5;">
<b>state &times; in_delaware</b><br>
<span style="color:#d62728;">&#9679;</span> NM (Delaware)
&nbsp; <span style="color:#ff9896;">&#9679;</span> NM (other)<br>
<span style="color:#1f77b4;">&#9679;</span> TX (Delaware)
&nbsp; <span style="color:#aec7e8;">&#9679;</span> TX (other)<br>
<span style="color:#555;">size &prop; RH_mean &middot; other layers in control box</span>
</div>'''
fmap.get_root().html.add_child(folium.Element(legend))

fmap.save("permian_sites_map.html")
print(f"{len(pts)} sites plotted -> permian_sites_map.html")
fmap


## Notes

- **Polygons vs points.** Footprints are ~280 m, invisible at basin zoom — that layer is for inspecting
  individual sites, not the overview. The point layers carry the basin-scale story.
- **Sizing / coloring.** Swap `RH_mean` for `Area_mean`, `N_dtct`, or the shapefile's `area`; recolor the
  temporal layer by *first* year or by persistence (count of years detected) instead of last year.
- **Caveat on the temporal view.** The annual columns stop at 2021 and reflect the 2012–2020 catalog vintage,
  so "last seen 2021" means *still present at series end*, not *survived the NM rule* — the catalog can't yet
  show post-2021 disappearance. Treat it as descriptive, not as the decoupling test itself.


In [ ]:
# === Monthly per-clear-look flaring heatmap, with a time slider =============
# Reads the folder of monthly-aggregate CSVs, computes mean radiant heat per
# clear look, and renders a month-by-month map you scrub with a slider/play.
#   pip install plotly pandas numpy
from pathlib import Path
import pandas as pd, numpy as np
import plotly.express as px

AGG_DIR = Path("./VNF_SITES_AGGREGATED/")        # <-- folder of per-site monthly CSVs

# one long table: every site x every month
mon = pd.concat([pd.read_csv(f) for f in sorted(AGG_DIR.glob("*.csv"))],
                ignore_index=True)

# the averaging metric: radiant heat per clear look (MW).
# clear looks = all detections + clear-sky non-detections (cloud mask 0 or 1).
# a month with only cloudy looks carries no information -> NaN, NOT zero.
mon["n_clear_look"] = mon["n_detect"] + mon["n_nondet_cm0"] + mon["n_nondet_cm1"]
mon["rh_per_clear_look"] = np.where(
    mon["n_clear_look"] > 0, mon["rh_sum"] / mon["n_clear_look"], np.nan)

# overpass lat/lon jitter slightly month to month -> pin each site to its median
xy = mon.groupby("flare_id")[["lat", "lon"]].median()
mon["lat"] = mon["flare_id"].map(xy["lat"])
mon["lon"] = mon["flare_id"].map(xy["lon"])

# drop only-cloudy months so "no data" never renders as "zero flaring"
plot = mon.dropna(subset=["rh_per_clear_look"]).sort_values("year_month")

# FIXED color range across frames, or months aren't comparable; cap at the
# 95th pct so a few huge flares don't wash out the rest.
vmax = float(plot["rh_per_clear_look"].quantile(0.95))
months = sorted(plot["year_month"].unique())

fig = px.scatter_map(            # plotly < 5.24: px.scatter_mapbox + mapbox_style="carto-positron"
    plot, lat="lat", lon="lon",
    color="rh_per_clear_look", range_color=(0, vmax),
    color_continuous_scale="Inferno",
    animation_frame="year_month", category_orders={"year_month": months},
    hover_name="flare_id",
    hover_data={"rh_per_clear_look": ":.3f", "n_clear_look": True,
                "lat": False, "lon": False, "year_month": False},
    zoom=6, height=650,
    labels={"rh_per_clear_look": "mean RH / clear look (MW)"},
)
fig.update_layout(map_style="carto-positron", margin=dict(l=0, r=0, t=30, b=0))
fig.update_traces(marker={"size": 8})
try:                              # slower autoplay so a ~170-frame run is watchable
    fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 400
except Exception:
    pass
fig.write_html("monthly_flaring_ylord1.html", include_plotlyjs="cdn")

# heavy-tailed? swap to a log view: 
#   plot["log_rh"] = np.log1p(plot["rh_per_clear_look"])
#   then color="log_rh" and remove range_color

In [ ]:
# --- corrected point slider: zeros recede in BOTH color and size ---
import numpy as np, plotly.express as px

plot = mon.dropna(subset=["rh_per_clear_look"]).sort_values("year_month").copy()
plot["mksize"] = np.sqrt(plot["rh_per_clear_look"])      # 0 -> ~invisible, high -> big
vmax   = float(plot["rh_per_clear_look"].quantile(0.95))
months = sorted(plot["year_month"].unique())

fig = px.scatter_map(
    plot, lat="lat", lon="lon",
    color="rh_per_clear_look", range_color=(0, vmax),
    color_continuous_scale="YlOrRd",                     # pale low end -> recedes; deep red -> pops
    size="mksize", size_max=14,
    animation_frame="year_month", category_orders={"year_month": months},
    hover_name="flare_id", zoom=6, height=650,
    labels={"rh_per_clear_look": "mean RH / clear look (MW)"},
)
fig.update_layout(map_style="carto-positron", margin=dict(l=0, r=0, t=30, b=0))
fig.update_traces(marker={"opacity": 0.85})
try: fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 400
except Exception: pass
fig.write_html("monthly_flaring_ylord2.html", include_plotlyjs="cdn")

# want quiet sites GONE rather than tiny?  plot = plot[plot.rh_per_clear_look > 0]
# want them faintly visible to show "present but not flaring"?  plot["mksize"] = np.sqrt(plot.rh_per_clear_look) + 0.15

In [ ]:
# --- smoothed density heatmap (impressionistic; read the caveats) ---
import plotly.express as px
plot = mon.dropna(subset=["rh_per_clear_look"]).sort_values("year_month").copy()
months = sorted(plot["year_month"].unique())
cmax = float(plot["rh_per_clear_look"].quantile(0.98))

fig = px.density_map(
    plot, lat="lat", lon="lon", z="rh_per_clear_look",
    radius=25,                                           # <-- a pixel knob, NOT physical
    range_color=(0, cmax), color_continuous_scale="Inferno",
    animation_frame="year_month", category_orders={"year_month": months},
    zoom=6, height=650,
)
fig.update_layout(map_style="carto-positron", margin=dict(l=0, r=0, t=30, b=0))
fig.write_html("monthly_flaring_ylord3.html", include_plotlyjs="cdn")